<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part A: Foundations and Data Exploration</h2>
<h2>Notebook A01: Loading and Manipulating the Data</h2>
</div>

This notebook covers the basics you need before moving on to visualisation and more advanced analysis: what a time series looks like in pandas, how to load one, and the handful of operations that come up in almost every project.

---

**Contents**

1. [Imports](#1.-Imports)
2. [Loading the Data](#2.-Loading-the-Data)
3. [Wide, Long, and Compact Formats](#3.-Wide,-Long,-and-Compact-Formats)
4. [Resampling](#4.-Resampling)
5. [Indexing and Slicing](#5.-Indexing-and-Slicing)
6. [Rolling Windows](#6.-Rolling-Windows)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports">1. Imports</h3>
</div>

We only need `pandas` for this notebook. `nb_config` gives us the dataset paths so we never need to hard-code them.

> **Datetime handling.** This notebook focuses on time series operations, not on parsing dates from raw strings. If you need a refresher on that, the resources below are a good starting point.
>
> | Source | Reference | Notes |
> | ------ | --------- | ----- |
> | [GitHub](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks_v1) | Python Data Science Handbook (Jake VanderPlas) | Notebook 3.11 focuses specifically on time series in pandas. |
> | [GitHub](https://github.com/PacktPublishing/Modern-Time-Series-Forecasting-with-Python-2E/tree/main/notebooks/Chapter02) | Modern Time Series Forecasting with Python, 2nd ed. (Manu Joseph) | Chapter 2 walks through the basics on a real dataset. |
> | [Kaggle](https://www.kaggle.com/code/parulpandey/getting-started-with-time-series-using-pandas) | Getting Started with Time Series Using Pandas (Parul Pandey) | A practical notebook covering the most common datetime operations. |
> | [Medium](https://medium.com/@noorfatimaafzalbutt/working-with-dates-and-times-in-pandas-a-comprehensive-guide-fda47929ace4) | Working with Dates and Times in Pandas (Noor Fatima) | A code-heavy guide to pandas datetime features. |

In [ ]:
import pandas as pd

import nb_config

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Loading-the-Data">2. Loading the Data</h3>
</div>

We use the CDC regional air temperature dataset throughout this notebook. It contains monthly mean air temperatures for 17 German regions from 1881 to the present, sourced from the [Deutscher Wetterdienst](https://www.dwd.de). Each column is a region; each row is one month.

> Each notebook uses one or two datasets as worked examples. You are free to follow along with those or swap in any dataset you prefer.

If you have not yet downloaded and prepared the dataset, run [F01b - Preparing the CDC dataset](../notebooks/F01b_Preparing_CDC_dataset.ipynb) first.

In [ ]:
df = pd.read_parquet(nb_config.CDC_TEMP_PATH)
df.head()

#### The DatetimeIndex

Notice that the index is a `DatetimeIndex`. This is the standard way to work with time series in pandas. It is what enables time-aware operations like resampling, slicing by date string, and rolling windows. If you load a dataset and the index is still a plain integer or a string, the first thing to do is convert it:

```python
df.index = pd.to_datetime(df.index)
```

Or, if the date is in a column rather than the index:

```python
df = df.set_index(pd.to_datetime(df["date_column"]))
```

Our CDC dataset already has a proper `DatetimeIndex` after the preparation step, so we can get straight to work.

In [ ]:
# Shape, dtypes, and missing value count at a glance
df.info()

In [ ]:
# Summary statistics for all regions
df.describe().T

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Wide,-Long,-and-Compact-Formats">3. Wide, Long, and Compact Formats</h3>
</div>

When a dataset contains several time series, there are a few common ways to organise it. You will run into all three in practice, so it is worth knowing how to convert between them.

| Format | Structure | Typical use |
|--------|-----------|-------------|
| **Wide** | One row per timestamp, one column per series | Most pandas operations, plotting |
| **Long** | One row per (timestamp, series) pair | Seaborn, some ML libraries, databases |
| **Compact** | One row per series, with the full value array stored as a single cell | Some specialised forecasting libraries |

Our CDC dataset is in wide format. We will use it to show all the conversions.

#### Wide to long

In [ ]:
df_long = pd.melt(
    df,
    ignore_index=False,       # keep the DatetimeIndex
    value_vars=df.columns,
    var_name="region",
    value_name="temperature",
)
df_long.index.name = "date"
df_long = df_long.sort_index()
df_long.head(10)

#### Long to wide

`pivot_table` reverses the melt. Each unique value in the `region` column becomes a column again.

In [ ]:
df_wide_from_long = df_long.pivot_table(
    index=df_long.index,
    columns="region",
    values="temperature",
)
df_wide_from_long.columns.name = None   # drop the 'region' label from the column axis
df_wide_from_long.head()

#### Compact format

Some forecasting libraries (e.g. GluonTS, certain Darts loaders) expect a compact layout: one row per series, where each row stores the metadata (start timestamp, frequency, number of observations) alongside the actual values as a single array. This is less common for data exploration but useful to recognise.

```bash
             Start       Frequency  n_Elements  Values
Deutschland  1881-01-01  1MS        1736        [-5.36, -2.73, ...]
Bayern       1881-01-01  1MS        1736        [-6.51, -3.62, ...]
...
```

The conversion from wide to compact is straightforward but we will not need it in this course until Part D. We show it here for completeness.

In [ ]:
df_compact = pd.DataFrame(
    {
        "Start": df.apply(lambda col: col.first_valid_index()),
        "Frequency": "1MS",
        "n_Elements": df.shape[0],
        "Values": [df[col].values for col in df.columns],
    },
    index=df.columns,
)
df_compact

And back to wide:

In [ ]:
series_dict = {
    series_name: pd.Series(
        data=row["Values"],
        index=pd.date_range(start=row["Start"], periods=row["n_Elements"], freq=row["Frequency"]),
    )
    for series_name, row in df_compact.iterrows()
}

df_wide_from_compact = pd.DataFrame(series_dict)
df_wide_from_compact.index.name = "date"
df_wide_from_compact.head()

**Exercise.** Convert `df_long` directly to compact format without going through wide first. Each group in `df_long` corresponds to one series.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Resampling">4. Resampling</h3>
</div>

Resampling changes the frequency of a time series.

**Downsampling** aggregates multiple observations into a single one, for example, going from monthly to yearly by taking the mean. You can use any aggregation function: `mean`, `sum`, `min`, `max`, `median`.

**Upsampling** goes the other way, introducing new time points between existing ones. The new points start out as `NaN` and need to be filled in. We cover the filling strategies in the missing data notebook.

The `resample` method takes a frequency alias string. Some common ones:

| Alias | Frequency |
|-------|-----------|
| `"h"` | Hourly |
| `"D"` | Daily |
| `"W"` | Weekly |
| `"MS"` | Month start |
| `"QS"` | Quarter start |
| `"YS"` | Year start |

In [ ]:
# Downsample to quarterly frequency — mean temperature per quarter
quarter_df = df.resample("QS").mean()
print(f"Monthly shape:   {df.shape}")
print(f"Quarterly shape: {quarter_df.shape}")
quarter_df.head()

In [ ]:
# Downsample to yearly frequency
year_df = df.resample("YS").mean()
print(f"Yearly shape: {year_df.shape}")
year_df.head()

In [ ]:
# Upsample to daily frequency — new rows are NaN until filled
daily_df = df.resample("D").asfreq()
print(f"Daily shape: {daily_df.shape}")
print(f"Missing values: {daily_df.isnull().sum().sum()}")
daily_df.head(10)

**Exercise.** Resample the CDC dataset to 10-year intervals and compute the maximum temperature recorded in each decade for `Deutschland`. Which decade was the warmest on record?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Indexing-and-Slicing">5. Indexing and Slicing</h3>
</div>

With a `DatetimeIndex`, selecting a time range is just a slice. Pandas understands partial date strings, so you can write `'2015'` instead of `'2015-01-01'` and it will match the whole year. A few examples:

```python
df['2015']                  # all of 2015
df['2015-06']               # June 2015 only
df['2010':'2020']           # 2010 through 2020 inclusive
df[:'1900']                 # everything up to the end of 1900
df['2000':]                 # everything from 2000 onwards
```

In [ ]:
# Last 10 years in the dataset
last_10y = df['2015':'2025']
print(f"Shape: {last_10y.shape}")
last_10y.head()

In [ ]:
# Selecting a single region
deutschland = df['Deutschland']
deutschland.head()

In [ ]:
# Selecting multiple regions
df[['Deutschland', 'Bayern', 'Schleswig-Holstein']].head()

**Exercise.** Select all observations from January across all years (i.e. every row where the month is 1). What is the coldest January on record for `Deutschland`?

In [ ]:
# Hint: df.index.month gives you the month number for each row
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Rolling-Windows">6. Rolling Windows</h3>
</div>

A rolling window computes a statistic over a sliding window of fixed size, moving one step at a time. It is one of the most common ways to smooth a noisy series or to generate lag-based features for a model.

The key parameters:

- `window`: the number of observations in the window.
- `min_periods`: the minimum number of non-NaN observations required to compute a result. If fewer observations are available (e.g. at the start of the series), the output is NaN. Setting it to `1` means the computation starts from the very first row; leaving it at the default (equal to `window`) means the first `window - 1` rows will be NaN.

In [ ]:
# 3-month rolling mean:  smooths out month-to-month noise
# min_periods=1 means we get a value even for the first two rows
rolling_3m = df.rolling(window=3, min_periods=1).mean()
rolling_3m.head(6)

In [ ]:
# 12-month rolling standard deviation: captures how variable each year is
# min_periods=12 means the first 11 rows will be NaN
rolling_12m_std = df.rolling(window=12, min_periods=12).std()
rolling_12m_std.head(15)

**Exercise.** Compute a 12-month centred rolling mean for `Deutschland`. A centred window (`center=True`) places the window symmetrically around each observation rather than looking only backwards. Does the result look different from the standard rolling mean? What does centering trade off?

In [ ]:
# Your solution here


---

You now have the core tools for loading and shaping time series data in pandas. The next notebook puts these to work visually: plotting the series, spotting seasonal patterns, and identifying anything unusual before you start building models.